In [1]:
"""
This script is used to compare the imputed vs the non-imputed ATAC matrices

authors: Roy Oelen
"""


'\nThis script is used to compare the imputed vs the non-imputed ATAC matrices\n\nauthors: Roy Oelen\n'

In [2]:
from scipy import sparse, io, stats
import numpy as np
import cupy as cp
import subprocess as sp
import os
import gc
import pandas as pd

In [3]:
class MtxGpuStats:
    
    def __init__(self, input_array):
        self.input_array = input_array
        self.row_means = None
        self.centered_array = None
        self.row_std_devs = None
        self.__calc_stats__()

    def __calc_stats__(self):
        # Calculate the mean along each row (axis=0)
        self.row_means = cp.mean(self.input_array, axis=1)

        # calculate the centered axis
        self.centered_array = self.input_array - self.row_means[:, cp.newaxis]

        # calculate the standard deviations
        self.row_std_devs = cp.std(self.input_array, axis=1, ddof=1)


In [4]:
def get_gpu_memory():
    command = "nvidia-smi --query-gpu=memory.free --format=csv"
    memory_free_info = sp.check_output(command.split()).decode('ascii').split('\n')[:-1][1:]
    memory_free_values = [int(x.split()[0]) for i, x in enumerate(memory_free_info)]
    return memory_free_values
    

def mtx_to_gpustats(mtx, rank=False):
    # convert to numpy array
    input_array_np = mtx.toarray()
    # if asked to rank, do so
    if rank:
        # rank data
        input_array_np_ranked = stats.rankdata(input_array_np, axis = 1)
        # assign to new variable
        input_array_np = input_array_np_ranked
        # clear memory
        del input_array_np_ranked
        
    # convert to cupy array
    input_array = cp.array(input_array_np, dtype=cp.float64)
    
    # put into object
    mtx_gpu_stats = MtxGpuStats(input_array)

    # return the result
    return mtx_gpu_stats

def calc_coeffs_per_pair(mtx1, mtx2, rank=False):
    m1 = mtx_to_gpustats(mtx1, rank)
    m2 = mtx_to_gpustats(mtx2, rank)
    covariance = cp.sum(m1.centered_array*m2.centered_array,axis=1) / m1.input_array.shape[1]
    std_products = m1.row_std_devs * m2.row_std_devs
    corcoeff = covariance / std_products
    return corcoeff


def get_matrix_stats(mtx1, mtx2, features_1, features_2, rank=False, chunk_size_correlation = 1000):
    # get the regions that are in the nonimputed data
    mtx2_indices_in_mtx1 = [features_2.index(value) for value in features_1]
    # create empty numpy array for results
    correlations = np.empty(len(features_1))
    # and two for the sums of the imputed and non-imputed
    mtx1_region_sums = np.empty(len(features_1))
    mtx2_region_sums = np.empty(len(features_1))
    # loop
    for i in range(0, len(features_1), chunk_size_correlation):
        # get the window
        left_window = i
        right_window = i + chunk_size_correlation
        # the right window can't exceed the number of features
        if right_window > len(features_1):
            right_window = len(features_1)
        # get the non-imputed indices for those imputed features
        mtx2_indices_chunk = mtx2_indices_in_mtx1[left_window:right_window]
        # now correlate the imputed and non-imputed for this chunk
        coeffs_chunk = calc_coeffs_per_pair(mtx1[left_window:right_window, :], mtx2[mtx2_indices_chunk, :], rank = True)
        # set these values in the correlations array
        correlations[left_window:right_window] = coeffs_chunk.get().reshape(-1)
        # calculate the sums of the regions as well
        mtx1_region_sums_chunk = mtx1[left_window:right_window, :].sum(axis=1)
        mtx2_region_sums_chunk = mtx2[mtx2_indices_chunk, :].sum(axis=1)
        # add to the numpy arrays we created
        mtx1_region_sums[left_window:right_window] = mtx1_region_sums_chunk.reshape(-1)
        mtx2_region_sums[left_window:right_window] = mtx2_region_sums_chunk.reshape(-1)
    # return result
    return np.array([correlations, mtx1_region_sums, mtx2_region_sums])
    

In [5]:
# locations of the imputed and non-imputed atac matrices
mtx_imputed_0_20000_loc = ''.join(['/groups/umcg-franke-scrna/tmp02/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'matrix_0_20000.mtx.gz'])
mtx_nonimputed_loc = ''.join(['/groups/umcg-franke-scrna/tmp02/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'nonimputed_matrix.mtx.gz'])

In [6]:
# load the first matrix
mtx_imputed_0_20000 = io.mmread(mtx_imputed_0_20000_loc).tocsr()

In [7]:
# load the second matrix
mtx_nonimputed = io.mmread(mtx_nonimputed_loc).tocsr()

In [8]:
# load the features file forthe imputed matrix
features_imputed_0_20000_loc = ''.join(['/groups/umcg-franke-scrna/tmp02/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'features_0_20000.tsv.gz'])
features_imputed_0_20000 = pd.read_csv(features_imputed_0_20000_loc, header=None)[0].tolist()

In [9]:
# load the features file forthe imputed matrix
features_nonimputed_loc = ''.join(['/groups/umcg-franke-scrna/tmp02/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'regiondata.tsv.gz'])
features_nonimputed = pd.read_csv(features_nonimputed_loc, header=0, sep = '\t')['name'].tolist()

In [10]:
get_gpu_memory()

[45525]

In [11]:
# get the regions that are in the nonimputed data
non_imputed_indices_in_imputed_0_20000 = [features_nonimputed.index(value) for value in features_imputed_0_20000]

In [ ]:
# now loop through chunks of the regions
chunk_size_correlation = 1000
# create empty numpy array for results
correlations = np.empty(len(features_imputed))
# and two for the sums of the imputed and non-imputed
imputed_region_sums = np.empty(len(features_imputed))
nonimputed_region_sums = np.empty(len(features_imputed))
# loop
for i in range(0, len(features_imputed), chunk_size_correlation):
    # get the window
    left_window = i
    right_window = i + chunk_size_correlation
    # the right window can't exceed the number of features
    if right_window > len(features_imputed):
        right_window = len(features_imputed)
    # get the non-imputed indices for those imputed features
    non_imputed_indices_chunk = non_imputed_indices_in_imputed[left_window:right_window]
    # now correlate the imputed and non-imputed for this chunk
    coeffs_chunk = calc_coeffs_per_pair(mtx_imputed[left_window:right_window, :], mtx_nonimputed[non_imputed_indices_chunk, :], rank = True)
    # set these values in the correlations array
    correlations[left_window:right_window] = coeffs_chunk.get().reshape(-1)
    # calculate the sums of the regions as well
    imputed_region_sums_chunk = mtx_imputed[left_window:right_window, :].sum(axis=1)
    nonimputed_region_sums_chunk = mtx_nonimputed[non_imputed_indices_chunk, :].sum(axis=1)
    # add to the numpy arrays we created
    imputed_region_sums[left_window:right_window] = imputed_region_sums_chunk.reshape(-1)
    nonimputed_region_sums[left_window:right_window] = nonimputed_region_sums_chunk.reshape(-1)


In [12]:
# get the stats
mtx_stats_0_20000 = get_matrix_stats(mtx_imputed_0_20000, mtx_nonimputed, features_imputed_0_20000, features_nonimputed, rank = True)

In [21]:
# turn results into dataframe
mtx_stats_0_20000_df = pd.DataFrame(data = mtx_stats_0_20000.transpose(), columns = ['correlation', 'original_sum', 'imputed_sum'])
# write to a file
mtx_stats_0_20000_df_loc = ''.join(['/groups/umcg-franke-scrna/tmp02/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'stats_0_20000.tsv.gz'])
mtx_stats_0_20000_df.to_csv(mtx_stats_0_20000_df_loc, header = True, index = None, sep = '\t', compression = 'gzip')

In [22]:
# clear up the memory
del mtx_imputed_0_20000

In [23]:
features_imputed_20000_40000_loc = ''.join(['/groups/umcg-franke-scrna/tmp02/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'features_20000_40000.tsv.gz'])
features_imputed_20000_40000 = pd.read_csv(features_imputed_20000_40000_loc, header=None)[0].tolist()
non_imputed_indices_in_imputed_20000_40000 = [features_nonimputed.index(value) for value in features_imputed_20000_40000]
mtx_imputed_20000_40000_loc = ''.join(['/groups/umcg-franke-scrna/tmp02/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'matrix_20000_40000.mtx.gz'])
mtx_imputed_20000_40000 = io.mmread(mtx_imputed_20000_40000_loc).tocsr()
mtx_imputed_20000_40000_loc = ''.join(['/groups/umcg-franke-scrna/tmp02/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'matrix_20000_40000.mtx.gz'])
mtx_stats_20000_40000 = get_matrix_stats(mtx_imputed_20000_40000, mtx_nonimputed, features_imputed_20000_40000, features_nonimputed, rank = True)
mtx_stats_20000_40000_df = pd.DataFrame(data = mtx_stats_20000_40000.transpose(), columns = ['correlation', 'original_sum', 'imputed_sum'])
mtx_stats_20000_40000_df_loc = ''.join(['/groups/umcg-franke-scrna/tmp02/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'stats_20000_40000.tsv.gz'])
mtx_stats_20000_40000_df.to_csv(mtx_stats_20000_40000_df_loc, header = True, index = None, sep = '\t', compression = 'gzip')
del mtx_imputed_20000_40000

In [ ]:
features_imputed_40000_60000_loc = ''.join(['/groups/umcg-franke-scrna/tmp02/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'features_40000_60000.tsv.gz'])
features_imputed_40000_60000 = pd.read_csv(features_imputed_40000_60000_loc, header=None)[0].tolist()
non_imputed_indices_in_imputed_40000_60000 = [features_nonimputed.index(value) for value in features_imputed_40000_60000]
mtx_imputed_40000_60000_loc = ''.join(['/groups/umcg-franke-scrna/tmp02/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'matrix_40000_60000.mtx.gz'])
mtx_imputed_40000_60000 = io.mmread(mtx_imputed_40000_60000_loc).tocsr()
mtx_imputed_40000_60000_loc = ''.join(['/groups/umcg-franke-scrna/tmp02/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'matrix_40000_60000.mtx.gz'])
mtx_stats_40000_60000 = get_matrix_stats(mtx_imputed_40000_60000, mtx_nonimputed, features_imputed_40000_60000, features_nonimputed, rank = True)
mtx_stats_40000_60000_df = pd.DataFrame(data = mtx_stats_40000_60000.transpose(), columns = ['correlation', 'original_sum', 'imputed_sum'])
mtx_stats_40000_60000_df_loc = ''.join(['/groups/umcg-franke-scrna/tmp02/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'stats_40000_60000.tsv.gz'])
mtx_stats_40000_60000_df.to_csv(mtx_stats_40000_60000_df_loc, header = True, index = None, sep = '\t', compression = 'gzip')
del mtx_imputed_40000_60000

In [ ]:
features_imputed_60000_80000_loc = ''.join(['/groups/umcg-franke-scrna/tmp02/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'features_60000_80000.tsv.gz'])
features_imputed_60000_80000 = pd.read_csv(features_imputed_60000_80000_loc, header=None)[0].tolist()
non_imputed_indices_in_imputed_60000_80000 = [features_nonimputed.index(value) for value in features_imputed_60000_80000]
mtx_imputed_60000_80000_loc = ''.join(['/groups/umcg-franke-scrna/tmp02/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'matrix_60000_80000.mtx.gz'])
mtx_imputed_60000_80000 = io.mmread(mtx_imputed_60000_80000_loc).tocsr()
mtx_imputed_60000_80000_loc = ''.join(['/groups/umcg-franke-scrna/tmp02/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'matrix_60000_80000.mtx.gz'])
mtx_stats_60000_80000 = get_matrix_stats(mtx_imputed_60000_80000, mtx_nonimputed, features_imputed_60000_80000, features_nonimputed, rank = True)
mtx_stats_60000_80000_df = pd.DataFrame(data = mtx_stats_60000_80000.transpose(), columns = ['correlation', 'original_sum', 'imputed_sum'])
mtx_stats_60000_80000_df_loc = ''.join(['/groups/umcg-franke-scrna/tmp02/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'stats_60000_80000.tsv.gz'])
mtx_stats_60000_80000_df.to_csv(mtx_stats_60000_80000_df_loc, header = True, index = None, sep = '\t', compression = 'gzip')
del mtx_imputed_60000_80000